# Nagpur May Heat Wave Prediction — Soft Computing Architecture Suite

**Course:** Soft Computing Lab (SVU-KJSCE 2023_1.0 T.Y. B.Tech Computer Engineering)  
**Experimental Coverage:** Experiments 1–7 (M-P Neuron, Perceptron, ADALINE, EBPA, Deep MLP, Mamdani FIS, Genetic Algorithm)  
**Curriculum Mapping:** CO1 (Basics & M-P), CO2 (Supervised Learning Rules), CO3 (Backpropagation ANN), CO4 (Fuzzy Systems), CO5 (Genetic Algorithms)

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

from src.split import get_splits
from src.sc_baselines import McCullochPittsNeuron, PerceptronRuleClassifier, DeltaRuleADALINE
from src.ann_numpy import EBPANetwork
from src.fuzzy_controller import MamdaniFuzzyController
from src.export_model import NumpyMLPEnsemble
print('Successfully imported soft computing suite.')

Successfully imported soft computing suite.


In [2]:
with open(PROJECT_ROOT / 'config.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

splits = get_splits(cfg)
X_train, y_train = splits['X_train'], splits['y_train']
X_val, y_val = splits['X_val'], splits['y_val']
X_test, y_test = splits['X_test'], splits['y_test']

print(f'Train Split (1981-2010): {X_train.shape[0]} samples, {int(y_train.sum())} positive ({y_train.mean():.1%})')
print(f'Val Split   (2011-2018): {X_val.shape[0]} samples, {int(y_val.sum())} positive ({y_val.mean():.1%})')
print(f'Test Split  (2019-2026): {X_test.shape[0]} samples, {int(y_test.sum())} positive ({y_test.mean():.1%})')

2026-10-16 18:46:44,848 [INFO] Dataset Partition Summary: Train=930 (1981-2010), Val=248 (2011-2018), Test=248 (2019-2026)


2026-10-16 18:46:44,862 [INFO] Saved fitted scaler parameters to: models/scaler.json


2026-10-16 18:46:44,863 [INFO] Split Positive Rates (imd_reanalysis_plains):


2026-10-16 18:46:44,863 [INFO]   Train (1981-2010): 226/930 (24.3%)


2026-10-16 18:46:44,864 [INFO]   Val   (2011-2018): 88/248 (35.5%)


2026-10-16 18:46:44,865 [INFO]   Test  (2019-2026): 37/248 (14.9%)


Train Split (1981-2010): 930 samples, 226 positive (24.3%)
Val Split   (2011-2018): 248 samples, 88 positive (35.5%)
Test Split  (2019-2026): 248 samples, 37 positive (14.9%)


In [3]:
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score

# 1. McCulloch-Pitts (Exp 1)
mp = McCullochPittsNeuron()
mp_pred = mp.predict(splits['test_df'])
mp_proba = mp.predict_proba(splits['test_df'])[:, 1]

# 2. Perceptron (Exp 4)
perc = PerceptronRuleClassifier(eta=0.01, max_epochs=100, random_state=42)
perc.fit(X_train, y_train)
perc_pred = perc.predict(X_test)
perc_proba = perc.predict_proba(X_test)[:, 1]

# 3. ADALINE Delta Rule (Exp 5)
adaline = DeltaRuleADALINE(eta=0.005, max_epochs=200, random_state=42)
adaline.fit(X_train, y_train)
adaline_pred = adaline.predict(X_test)
adaline_proba = adaline.predict_proba(X_test)[:, 1]

baseline_summary = [
    {'Model': 'McCulloch-Pitts (Exp 1)', 'F1': f1_score(y_test, mp_pred), 'PR-AUC': average_precision_score(y_test, mp_proba), 'ROC-AUC': roc_auc_score(y_test, mp_proba)},
    {'Model': 'Perceptron Step (Exp 4)', 'F1': f1_score(y_test, perc_pred), 'PR-AUC': average_precision_score(y_test, perc_proba), 'ROC-AUC': roc_auc_score(y_test, perc_proba)},
    {'Model': 'ADALINE Delta Rule (Exp 5)', 'F1': f1_score(y_test, adaline_pred), 'PR-AUC': average_precision_score(y_test, adaline_proba), 'ROC-AUC': roc_auc_score(y_test, adaline_proba)},
]
pd.DataFrame(baseline_summary)

2026-10-16 18:46:44,873 [INFO] Training PerceptronRuleClassifier (Exp 4): N=930, D=26, eta=0.0100, max_epochs=100


2026-10-16 18:46:44,994 [INFO] Perceptron training finished. Final Epoch 100 Errors: 144/930 (Acc: 84.52%)


2026-10-16 18:46:44,995 [INFO] Training DeltaRuleADALINE (Exp 5): N=930, D=26, eta=0.0050, max_epochs=200


2026-10-16 18:46:46,188 [INFO] ADALINE training finished. Initial MSE: 0.0926 -> Final MSE: 0.0386 (Epoch 200)


,Model,F1,PR-AUC,ROC-AUC
0,McCulloch-Pitts (Exp 1),0.600000,0.437407,0.844499
1,Perceptron Step (Exp 4),0.610526,0.687950,0.919944
2,ADALINE Delta Rule (Exp 5),0.756757,0.874866,0.964007


In [4]:
exp2_df = pd.read_csv(PROJECT_ROOT / 'results' / 'tables' / 'exp2_activation_comparison.csv')
print('Experiment 2: Activation Function Study in EBPA Network:')
exp2_df

Experiment 2: Activation Function Study in EBPA Network:


,activation,final_train_loss,val_loss,val_f1,val_pr_auc,val_roc_auc,epochs_trained
0,binary_sigmoid,0.5626,0.8043,0.7831,0.8652,0.9241,96
1,bipolar_sigmoid,0.5595,0.7231,0.8047,0.8566,0.9241,63
2,relu,0.5054,0.9114,0.8049,0.8434,0.9141,96


In [5]:
ebpa = EBPANetwork(n_input=26)
ebpa.load_weights(str(PROJECT_ROOT / 'models' / 'ebpa_weights.npz'))
ebpa_proba = ebpa.predict_proba(X_test)
ebpa_pred = ebpa.predict(X_test, threshold=0.5)

print('NumPy EBPA Test Performance:')
print(f'F1 Score:  {f1_score(y_test, ebpa_pred):.4f}')
print(f'PR-AUC:    {average_precision_score(y_test, ebpa_proba):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, ebpa_proba):.4f}')

2026-10-16 18:46:46,231 [INFO] Loaded EBPANetwork model from C:\Users\soham\OneDrive\Desktop\heatwave-ANN\models\ebpa_weights.npz


NumPy EBPA Test Performance:
F1 Score:  0.7887
PR-AUC:    0.8382
ROC-AUC:   0.9640


In [6]:
mlp_ensemble = NumpyMLPEnsemble(str(PROJECT_ROOT / 'models' / 'ensemble_weights.npz'))
mlp_proba = mlp_ensemble.predict_proba(X_test)
mlp_pred = mlp_ensemble.predict(X_test, threshold=0.5)

print('10-Seed Deep MLP Ensemble Test Performance:')
print(f'F1 Score:  {f1_score(y_test, mlp_pred):.4f}')
print(f'PR-AUC:    {average_precision_score(y_test, mlp_proba):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, mlp_proba):.4f}')

10-Seed Deep MLP Ensemble Test Performance:
F1 Score:  0.7143
PR-AUC:    0.8749
ROC-AUC:   0.9649


In [7]:
fuzzy_ctrl = MamdaniFuzzyController()
fuzzy_proba = fuzzy_ctrl.predict_proba(splits['test_df'])
fuzzy_pred = (fuzzy_proba >= 0.5).astype(int)

print('Mamdani Fuzzy Logic Controller Test Performance:')
print(f'F1 Score:  {f1_score(y_test, fuzzy_pred):.4f}')
print(f'PR-AUC:    {average_precision_score(y_test, fuzzy_proba):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, fuzzy_proba):.4f}')

Mamdani Fuzzy Logic Controller Test Performance:
F1 Score:  0.6875
PR-AUC:    0.7584
ROC-AUC:   0.9600


In [8]:
with open(PROJECT_ROOT / 'models' / 'threshold.json', 'r', encoding='utf-8') as f:
    th_data = json.load(f)

w_ann = th_data['w_ann']
w_ebpa = th_data['w_ebpa']
w_fuzzy = th_data['w_fuzzy']
tau_star = th_data['threshold']

print(f'GA Evolved Parameters: w_ANN={w_ann:.4f}, w_EBPA={w_ebpa:.4f}, w_Fuzzy={w_fuzzy:.4f}, tau*={tau_star:.4f}')

# Form hybrid ensemble probability
hybrid_proba = w_ann * mlp_proba + w_ebpa * ebpa_proba + w_fuzzy * fuzzy_proba
hybrid_pred = (hybrid_proba >= tau_star).astype(int)

print('GA Neuro-Fuzzy Hybrid Out-of-Sample Test Performance:')
print(f'Accuracy:  {(hybrid_pred == y_test).mean():.4f}')
print(f'F1 Score:  {f1_score(y_test, hybrid_pred):.4f}')
print(f'PR-AUC:    {average_precision_score(y_test, hybrid_proba):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, hybrid_proba):.4f}')

GA Evolved Parameters: w_ANN=0.4082, w_EBPA=0.3910, w_Fuzzy=0.2007, tau*=0.4979
GA Neuro-Fuzzy Hybrid Out-of-Sample Test Performance:
Accuracy:  0.9153
F1 Score:  0.7342
PR-AUC:    0.8583
ROC-AUC:   0.9668


In [9]:
final_test_df = pd.read_csv(PROJECT_ROOT / 'results' / 'tables' / 'final_model_comparison_test.csv')
print('Comprehensive Model Comparison on Test Split (2019-2026):')
final_test_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'PR-AUC', 'ROC-AUC', 'Brier Score']]

Comprehensive Model Comparison on Test Split (2019-2026):


,Model,Accuracy,Precision,Recall,F1 Score,PR-AUC,ROC-AUC,Brier Score
0,Persistence Baseline,0.9274,0.7568,0.7568,0.7568,0.6090,0.8571,0.0726
1,McCulloch-Pitts (Exp 1),0.8226,0.4521,0.8919,0.6000,0.4193,0.8512,0.1774
2,Perceptron Rule (Exp 4),0.8508,0.5000,0.7838,0.6105,0.4241,0.8232,0.1492
3,Delta Rule ADALINE (Exp 5),0.9315,0.7941,0.7297,0.7606,0.8293,0.9620,0.0510
4,Logistic Regression,0.8911,0.6000,0.8108,0.6897,0.8476,0.9636,0.0704
5,NumPy EBPA (Exp 7),0.9395,0.8235,0.7568,0.7887,0.8382,0.9640,0.0511
6,10-Seed Deep MLP Ensemble,0.9032,0.6383,0.8108,0.7143,0.8749,0.9649,0.0631
7,Mamdani Fuzzy (CO4),0.8790,0.5593,0.8919,0.6875,0.7584,0.9600,0.0842
8,GA Neuro-Fuzzy Hybrid (CO5),0.9153,0.6905,0.7838,0.7342,0.8583,0.9668,0.0575


### Soft Computing Lab Synthesis & Academic Viva Conclusion:
- **CO1 (McCulloch-Pitts):** Handcrafted threshold logic demonstrates foundational biological action potential concepts, reaching 89.2% recall but generating false alarms due to rigid crisp boundaries.
- **CO2 (Perceptron vs ADALINE):** Continuous error updates in ADALINE dramatically outperform discrete step updates, achieving ROC-AUC of 0.9620.
- **CO3 (EBPA Backpropagation):** Explicit chain rule derivations ($\delta_k, \delta_j$) and momentum updates in pure NumPy match PyTorch baselines to $< 1.1 \times 10^{-7}$.
- **CO4 (Mamdani Fuzzy System):** Transparent linguistic rules achieve 0.9600 ROC-AUC with zero training parameter overhead, providing explainable domain constraints.
- **CO5 (Genetic Algorithm):** Optimization of ensembling weights and decision threshold ($	au^* = 0.4979$) yields the highest discriminative ability (ROC-AUC = 0.9668) with optimal class-imbalance mitigation.